In [1]:
import os
import sys
sys.path.append("Config")
sys.path.append("Experiments")
import pandas as pd
from config import DATASETS, INPUT_SETTINGS, LABEL_SETTINGS, SCALE_THRESHOLD, ori_dir, ex_dir
from utils import load_splits

In [2]:
# Original splits (Data/<DATASET>/ori/{train,val,test}.jsonl)
for dataset in DATASETS:
    for split in ["train", "val", "test"]:
        df = pd.read_json(f"{ori_dir(dataset)}{split}.jsonl", lines=True, dtype={"id": str})
        print(f"{dataset} {split}: {len(df)} memes, labels: {df.label.value_counts().sort_index().to_dict()}")
    print()

FHM train: 8500 memes, labels: {0: 5481, 1: 3019}
FHM val: 640 memes, labels: {0: 393, 1: 247}
FHM test: 3000 memes, labels: {0: 1760, 1: 1240}

MAMI train: 9000 memes, labels: {0: 4500, 1: 4500}
MAMI val: 1000 memes, labels: {0: 500, 1: 500}
MAMI test: 1000 memes, labels: {0: 500, 1: 500}



In [3]:
# Label distribution used in each experiment setting
def print_stats(dataset, setting):
    try:
        train_df, val_df, test_df = load_splits(dataset, setting)
    except FileNotFoundError as e:
        print(f"({dataset}, {setting}) not generated yet: {e}\n")
        return

    print(f"({dataset}, {setting}) Train/Val/Test Set Label Counts:")
    for name, d in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
        print(f"{name}: {d.label.value_counts().sort_index().to_dict()}")
    if setting == "inputs":
        for name, d in [("Train", train_df), ("Validation", val_df)]:
            print(f"Augmented non-hateful memes in {name}: {d.img_path.str.contains('/ex/').sum()}")
    if setting.endswith("_scale"):
        print(f"Train scale distribution: {train_df.train_label.value_counts().sort_index().to_dict()}")
    print()

for dataset in DATASETS:
    for setting in INPUT_SETTINGS + LABEL_SETTINGS:
        print_stats(dataset, setting)
    print("#" * 50 + "\n")

(FHM, orig) Train/Val/Test Set Label Counts:
Train: {0: 5481, 1: 3019}
Validation: {0: 393, 1: 247}
Test: {0: 1760, 1: 1240}

(FHM, inputs) Train/Val/Test Set Label Counts:
Train: {0: 6756, 1: 3019}
Validation: {0: 433, 1: 247}
Test: {0: 1760, 1: 1240}
Augmented non-hateful memes in Train: 1275
Augmented non-hateful memes in Validation: 40

(FHM, simple_binary) Train/Val/Test Set Label Counts:
Train: {0: 5481, 1: 3019}
Validation: {0: 393, 1: 247}
Test: {0: 1760, 1: 1240}

(FHM, simple_scale) Train/Val/Test Set Label Counts:
Train: {0: 5290, 1: 1213}
Validation: {0: 375, 1: 50}
Test: {0: 1760, 1: 1240}
Train scale distribution: {0: 4398, 1: 415, 2: 272, 3: 120, 4: 85, 5: 103, 6: 310, 7: 374, 8: 295, 9: 131}

(FHM, category_binary) Train/Val/Test Set Label Counts:
Train: {0: 5481, 1: 3019}
Validation: {0: 393, 1: 247}
Test: {0: 1760, 1: 1240}

(FHM, category_scale) Train/Val/Test Set Label Counts:
Train: {0: 5298, 1: 1130}
Validation: {0: 377, 1: 44}
Test: {0: 1760, 1: 1240}
Train scale

### Labels: teacher scale scores vs binary ground truth

In [4]:
for dataset in DATASETS:
    for split in ["train", "val"]:
        path = f"{ex_dir(dataset)}ex_{split}_scale.jsonl"
        if not os.path.exists(path):
            print(f"{path} not generated yet")
            continue
        df = pd.read_json(path, lines=True, dtype={"id": str})
        for prompt in ["simple_scale", "category_scale"]:
            agree = (df.label == (df[prompt] >= SCALE_THRESHOLD).astype(int)) & df[prompt].notna()
            print(f"{dataset} {split} {prompt}: agreement {agree.sum()}/{len(df)} ({100*agree.mean():.2f}%), unparsable {df[prompt].isna().sum()}")
            display(pd.crosstab(df[prompt], df.label, margins=True))

FHM train simple_scale: agreement 6503/8500 (76.51%), unparsable 0


label,0,1,All
simple_scale,,,
0,4398,930,5328
1,415,181,596
2,272,306,578
3,120,167,287
4,85,222,307
5,27,103,130
6,88,310,398
7,50,374,424
8,20,295,315


FHM train category_scale: agreement 6428/8500 (75.62%), unparsable 0


label,0,1,All
category_scale,,,
0,4572,1055,5627
1,238,126,364
2,285,310,595
3,102,161,263
4,101,237,338
5,44,107,151
6,68,289,357
7,51,313,364
8,18,281,299


FHM val simple_scale: agreement 425/640 (66.41%), unparsable 0


label,0,1,All
simple_scale,,,
0,319,123,442
1,23,21,44
2,17,24,41
3,11,10,21
4,5,19,24
5,4,6,10
6,8,16,24
7,4,14,18
8,2,11,13


... (6 outputs left out)

MAMI val simple_scale: agreement 531/1000 (53.10%), unparsable 0


label,0,1,All
simple_scale,,,
0,446,296,742
1,22,57,79
2,17,73,90
3,4,25,29
4,4,11,15
5,1,4,5
6,4,23,27
7,1,7,8
8,1,4,5


MAMI val category_scale: agreement 516/1000 (51.60%), unparsable 0


label,0,1,All
category_scale,,,
0,459,310,769
1,15,34,49
2,12,86,98
3,7,32,39
4,1,16,17
5,3,10,13
6,1,4,5
7,2,6,8
8,0,2,2


### Inputs: meme typology of hateful memes (H/N for background, caption)

In [5]:
def label_map(x):
    if x is None or pd.isna(x):
        return None
    x = x.lower()
    return 1 if "yes" in x else (0 if "no" in x else None)

for dataset in DATASETS:
    for split in ["train", "val"]:
        path = f"Augmentation Pipeline/Outputs/{dataset}/Full/inputs/with_hatelabels_with_desc_{split}.jsonl"
        if not os.path.exists(path):
            print(f"{path} not generated yet")
            continue
        df = pd.read_json(path, lines=True, dtype={"id": str})
        d, t = df.hateLabelDesc.apply(label_map), df.hateLabelText.apply(label_map)
        counts = {typ: int(((d == a) & (t == b)).sum()) for typ, (a, b) in {"HH": (1, 1), "HN": (1, 0), "NH": (0, 1), "NN": (0, 0)}.items()}
        print(f"{dataset} {split}: {len(df)} hateful memes, {counts}, unparsable: {int((d.isna() | t.isna()).sum())}")

FHM train: 3019 hateful memes, {'HH': 24, 'HN': 27, 'NH': 1504, 'NN': 1464}, unparsable: 0
FHM val: 247 hateful memes, {'HH': 2, 'HN': 5, 'NH': 52, 'NN': 188}, unparsable: 0
MAMI train: 4500 hateful memes, {'HH': 6, 'HN': 12, 'NH': 921, 'NN': 3561}, unparsable: 0
MAMI val: 500 hateful memes, {'HH': 0, 'HN': 0, 'NH': 111, 'NN': 389}, unparsable: 0
